In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S007C001P015R001A008_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P016R001A027_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P008R001A023_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S005C001P017R001A009_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P007R001A008_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P008R001A009_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S005C001P010R001A008_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S007C001P017R001A015_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S007C001P019R001A008_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S004C001P008R001A015_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P019R001A008_rgb.avi
/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset/S006C001P016R001A015_

In [2]:
!git clone https://github.com/trishaShah-web/testing123.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 132 (delta 50), reused 103 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 116.70 KiB | 2.20 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/kaggle/working/repo


In [3]:
!pip install -q torchcodec timm einops decord ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 88.4 MB/s eta 0:00:00


In [4]:
!git pull

Already up to date.


In [5]:
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True | Tesla T4


In [6]:
import torch
try:
    torch.hub.load("facebookresearch/vjepa2", "vjepa2_vit_large", pretrained=True, trust_repo=True)
except Exception as e:
    print("expected, patching next:", e)

Downloading: "https://github.com/facebookresearch/vjepa2/zipball/main" to /root/.cache/torch/hub/main.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Downloading: "http://localhost:8300/vitl.pt" to /root/.cache/torch/hub/checkpoints/vitl.pt
expected, patching next: <urlopen error [Errno 111] Connection refused>


In [7]:
!sed -i 's|http://localhost:8300|https://dl.fbaipublicfiles.com/vjepa2|g' \
  /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py
!grep -n "VJEPA_BASE_URL" /root/.cache/torch/hub/facebookresearch_vjepa2_main/src/hub/backbones.py

8:# VJEPA_BASE_URL = "https://dl.fbaipublicfiles.com/vjepa2"
11:VJEPA_BASE_URL = "https://dl.fbaipublicfiles.com/vjepa2"
81:        url = VJEPA_BASE_URL + f"/{model_file}.pt"
146:        url = VJEPA_BASE_URL + f"/{model_file}.pt"
274:        url = VJEPA_BASE_URL + f"/{model_file}.pt"


In [8]:
import os; os.environ["PYTHONPATH"] = "."
SUBSET_170 = "/kaggle/input/datasets/trishi11/ntu-rgbd-s00567-subset"
!ls "$SUBSET_170" | wc -l   # expect 170

170


In [9]:
!python scripts/visualize_steering.py -h

usage: visualize_steering.py [-h] --target-clip TARGET_CLIP --anchor ANCHOR
                             --pca-basis PCA_BASIS [--blind-alpha BLIND_ALPHA]
                             [--output-dir OUTPUT_DIR]
                             ntu_root

Day-2-adjacent visualization: for ONE target clip, render Raw-vs-Blind
comparisons using the SAME encode/predict/steer code path
scripts/spike_blind_vs_raw.py already uses (inlined here, not routed through
pipeline/inference_loop.py's SteeringPipeline — that class still hardcodes
the pre-DEVIATIONS-#6 64-frame mask dims and would break on the real
16-frame config; see HANDOFF.md, flagged separately, not fixed here).

Two outputs, both purely latent-based (no pixel decoder anywhere):

1. PCA overlay (primary): predicted (Raw) and steered (Blind) target-region
   latents projected through a SHARED PCA basis (scripts/build_pca_basis.py,
   loaded here, never refit) -> RGB -> patch grid -> upsampled -> alpha-
   blended over the clip's own real 

In [10]:
!ls "$SUBSET_170" | grep -E '\.avi$|\.mp4$' | head -20

S004C001P003R001A008_rgb.avi
S004C001P003R001A009_rgb.avi
S004C001P003R001A015_rgb.avi
S004C001P003R001A023_rgb.avi
S004C001P003R001A027_rgb.avi
S004C001P007R001A008_rgb.avi
S004C001P007R001A009_rgb.avi
S004C001P007R001A015_rgb.avi
S004C001P007R001A023_rgb.avi
S004C001P007R001A027_rgb.avi
S004C001P008R001A008_rgb.avi
S004C001P008R001A009_rgb.avi
S004C001P008R001A015_rgb.avi
S004C001P008R001A023_rgb.avi
S004C001P008R001A027_rgb.avi
S004C001P020R001A008_rgb.avi
S004C001P020R001A009_rgb.avi
S004C001P020R001A015_rgb.avi
S004C001P020R001A023_rgb.avi
S004C001P020R001A027_rgb.avi


In [11]:
TARGET_CLIP = f"{SUBSET_170}/S004C001P003R001A023_rgb.avi"

In [12]:
!python scripts/build_semantic_anchor.py "$SUBSET_170" --target-clip "$TARGET_CLIP" --output anchor.pt

inferred from target clip: action=23 (hand waving), exclude_performer=[3], camera=1
device: cuda
pooling 32 reference clip(s) across performers [1, 4, 7, 8, 10, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27] for action 23 (hand waving), excluding performer [3], camera=1
reserved performer(s) [28] out of the anchor — their clips for this action remain available as disjoint probe-training material
loading encoder + predictor (torch.hub, first run will download)...
Using cache found in /root/.cache/torch/hub/facebookresearch_vjepa2_main
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Downloading: "https://dl.fbaipublicfiles.com/vjepa2/vitl.pt" to /root/.cache/torch/hub/checkpoints/vitl.pt
100%|██████████████████████████████████████| 4.78G/4.78G [01:58<00:00

In [13]:
!python scripts/build_pca_basis.py "$SUBSET_170" --from-anchor anchor.pt --output pca_basis.pt

reusing 32 reference clip(s) from anchor.pt as the PCA reference set
device: cuda
PCA reference set: 32 clip(s)
Using cache found in /root/.cache/torch/hub/facebookresearch_vjepa2_main
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
  encoded S004C001P007R001A023_rgb.avi (1024 target tokens)
  encoded S004C001P008R001A023_rgb.avi (1024 target tokens)
  encoded S004C001P020R001A023_rgb.avi (1024 target tokens)
  encoded S005C001P004R001A023_rgb.avi (1024 target tokens)
  encoded S

In [14]:
!python scripts/visualize_steering.py "$SUBSET_170" --target-clip "$TARGET_CLIP" --anchor anchor.pt --pca-basis pca_basis.pt

target: S004C001P003R001A023_rgb.avi -> action A23 (hand waving), performer P3, camera C1
blind_alpha = 0.3, device = cuda
loading encoder + predictor (torch.hub, first run will download)...
Using cache found in /root/.cache/torch/hub/facebookresearch_vjepa2_main
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
loaded shared PCA basis from pca_basis.pt (fit once, reused for every arm below)

encoding + predicting target clip S004C001P003R001A023_rgb.avi...
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
drift d = 1 - cos(predic

In [15]:
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
!cp *.pt /kaggle/working/checkpoints/ 2>/dev/null
!ls -R /kaggle/working/checkpoints/

/kaggle/working/checkpoints/:
anchor.pt  pca_basis.pt  viz

/kaggle/working/checkpoints/viz:
S004C001P003R001A023_rgb

/kaggle/working/checkpoints/viz/S004C001P003R001A023_rgb:
nn_retrieval_blind_side_by_side.png  pca_raw.png	   retrieval_blind.mp4
nn_retrieval_raw_side_by_side.png    pca_raw_vs_blind.png  retrieval_raw.mp4
pca_blind.png			     reference_frames

/kaggle/working/checkpoints/viz/S004C001P003R001A023_rgb/reference_frames:
S004C001P007R001A023_rgb_block0.png  S006C001P017R001A023_rgb_block0.png
S004C001P007R001A023_rgb_block1.png  S006C001P017R001A023_rgb_block1.png
S004C001P007R001A023_rgb_block2.png  S006C001P017R001A023_rgb_block2.png
S004C001P007R001A023_rgb_block3.png  S006C001P017R001A023_rgb_block3.png
S004C001P008R001A023_rgb_block0.png  S006C001P019R001A023_rgb_block0.png
S004C001P008R001A023_rgb_block1.png  S006C001P019R001A023_rgb_block1.png
S004C001P008R001A023_rgb_block2.png  S006C001P019R001A023_rgb_block2.png
S004C001P008R001A023_rgb_block3.png  S006C001P019